In [20]:
#Fine tuning a pretrained model
# Scenario: Fine-tune ResNet-50 pretrained on ImageNet
#  to classify 5 types of plant disease from 3,000 leaf images.

In [22]:
import torch
import torchvision
import torch.nn as nn
from torchvision import models , transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

In [23]:
model = models.resnet50(weights='IMAGENET1K_V2')
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [24]:
for param in model.parameters():
    param.requires_grad = False

In [25]:
num_classes = 5

In [26]:
model.fc = nn.Sequential(
     nn.Dropout(0.4),
     nn.Linear(model.fc.in_features, 256),
     nn.ReLU(),
     nn.Linear(256, num_classes) 
)

In [30]:
model.fc

Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=2048, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=5, bias=True)
)

In [27]:
for param in model.layer4.parameters():
    param.requires_grad = True

In [28]:
optimizer = torch.optim.Adam([

    # Lower learning rate for pretrained layer4
    {'params': model.layer4.parameters(), 'lr': 1e-4},

    # Higher learning rate for new classifier head
    {'params': model.fc.parameters(), 'lr': 1e-3},
])

In [29]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [15]:
criterion = nn.CrossEntropyLoss()

In [18]:
criterion

CrossEntropyLoss()

In [39]:
# 🎯 Scenario: Medical Image Classification
# You’re training a convolutional neural network (CNN) to detect pneumonia from chest X-rays.
# - Training accuracy: 95%
# - Validation accuracy: 74%
# At first glance, the model seems powerful — it almost perfectly classifies the training set. But the sharp drop in validation accuracy signals overfitting: the network has memorized the training images (specific pixel patterns, noise, or even hospital-specific artifacts) instead of learning generalizable features of pneumonia.

# ⚙️ Levers to Address Overfitting
# - Data Augmentation: Rotate, flip, and adjust brightness of X-rays to simulate variability.
# - Regularization: Apply dropout in dense layers or L2 weight decay.
# - Transfer Learning: Use a pretrained backbone (e.g., ResNet) to leverage generalized features.
# - Cross-validation: Ensure robustness across different patient subsets.
# - Early Stopping: Halt training when validation loss stops improving.

In [41]:
from torchvision import transforms
import torch.optim as optim
import torch
import torch.nn as nn
import numpy as np

In [44]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
     transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2
    ),

    transforms.RandomRotation(degrees=15),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],   # mean for RGB channels
        [0.229,0.224,0.225]    # standard deviation for RGB channels
    ),

    transforms.RandomErasing(p=0.2)    
])

train_tf

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.2)
    ColorJitter(brightness=(0.7, 1.3), contrast=(0.7, 1.3), saturation=(0.8, 1.2), hue=None)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    RandomGrayscale(p=0.1)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    RandomErasing(p=0.2, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0, inplace=False)
)

In [45]:
def mixup_data(x, y, alpha=0.4):

    # Sample mixing coefficient (lambda) from Beta distribution
    # Beta distribution ensures lambda is between 0 and 1
    lam = np.random.beta(alpha, alpha)

    # Randomly shuffle indices of the batch
    # This determines which image will be mixed with which
    idx = torch.randperm(x.size(0))

    # Create mixed images
    # New image = lam * imageA + (1-lam) * imageB
    mixed_x = lam * x + (1 - lam) * x[idx]

    # Return mixed images and both labels
    # Loss will be computed using both labels weighted by lambda
    return mixed_x, y, y[idx], lam


In [46]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,      # optimizer whose learning rate will be scheduled

    T_max = 50,     # number of epochs before the learning rate reaches minimum

    eta_min = 1e-6  # minimum learning rate value

)

In [47]:
criterion = nn.CrossEntropyLoss(

    label_smoothing = 0.1   # distributes a small probability mass to other classes

)

In [48]:
torch.nn.utils.clip_grad_norm_(

    model.parameters(),   # parameters whose gradients will be clipped

    max_norm = 1.0        # maximum allowed gradient norm

)

tensor(0.)